# Learnable node embeddings - is the R-GCN necessary, or just the only route for node identity?

This is the control that `ablation_encoder_swap.ipynb` identified as blocking its strongest reading, and
it is the single most important missing experiment in the study.

### The confound

`initialize_node_embedding` (`utils.py:1124`) creates node features as

```python
emb = nn.Parameter(torch.Tensor(g[ntype].num_nodes, n_inp), requires_grad = False)
nn.init.xavier_uniform_(emb)
g[ntype].inp = emb
```

**frozen** xavier-random vectors. In the `mlp` arm of the encoder-swap grid, a node embedding is
therefore a deterministic function of a fixed random vector through 20,200 *shared* parameters: the
model has **no learnable per-node capacity at all**, and the R-GCN is the only mechanism by which node
identity can reach the decoder. That is very likely why indication AUPRC fell to 0.5285 - near the 0.500
floor - rather than degrading gracefully.

So the encoder-swap result supports "replacing the R-GCN with an MLP destroys the model **as the model
is built**", but it does **not** support "message passing is what the model needs". Until per-node
capacity is restored, "the GNN is unnecessary" is not testable from that data.

### The fix

Replace the frozen random features with a **learnable embedding table** - `nn.Parameter` per node type,
registered on the model so `AdamW(self.model.parameters())` optimises it - and otherwise change nothing.

| arm | encoder | node features | message passing |
|---|---|---|---|
| `rgcn` | `HeteroRGCNLayer` x2 | frozen random | yes |
| `mlp` | `MLPLayer` x2 | frozen random | **no** |
| `emb_mlp` | `MLPLayer` x2 | **learnable** | **no** |
| `emb_rgcn` | `HeteroRGCNLayer` x2 | **learnable** | yes |

- **`emb_mlp` vs `mlp`** is the control: does the MLP recover once nodes can be identified?
- **`emb_mlp` vs `rgcn`** is the real question: with per-node capacity available to both, does message
  passing still buy anything?
- **`emb_rgcn` vs `rgcn`** asks whether the frozen features limit the published model too. Set
  `INCLUDE_EMB_RGCN = False` to halve the GPU cost if you only want the first two.

### Read the parameter counts honestly

The embedding table is **not** parameter-matched and cannot be. ~129k nodes x 100 dims is ~12.9M
parameters, an order of magnitude more than the R-GCN's 1.01M. That is inherent to the control: it is
restoring a *capability* (per-node identity) that the published model lacks, not equalising a budget.
So:

- If `emb_mlp` stays near the floor **even with 12.9M free parameters**, the conclusion is strong and
  clean: node identity is not the issue, and message passing really is load-bearing for indication.
- If `emb_mlp` recovers most of the gap, then the honest statement is that the R-GCN's contribution in
  the published model is substantially *a way of giving nodes identities*, and a plain embedding table
  does the same job - but the comparison is not at equal capacity and the write-up must say so.

Either outcome is reportable; the one thing that is not defensible is quoting `mlp` alone as evidence
about message passing. Exact counts are recorded per run as `encoder_params`, `node_emb_params` and
`n_params`.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))

import torch
import torch.nn as nn
from torch_scatter import scatter

import txgnn
from txgnn import TxData, TxGNN
from txgnn.model import HeteroRGCN, HeteroRGCNLayer
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit this notebook.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo  :', REPO_ROOT)
print('txgnn :', txgnn.__file__)
print('torch :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 - Configuration

Identical model and schedule to `ablation_encoder_swap.ipynb`, same 20 seeds, so `rgcn` and `mlp`
both import free from that grid and every delta is directly comparable to the published ones.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # metric learning stays ON in every arm
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES_DEFAULT = ['disease_disease', 'rev_disease_protein']

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_learnable_embeddings'
SWAP_ROOT   = REPO_ROOT / 'results' / 'ablation_encoder_swap'   # read-only: both references
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM   = 'rgcn'   # primary reference
REF_ARM_2 = 'mlp'    # the frozen-feature MLP whose collapse this notebook explains

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True
SAVE_MODELS       = False
IMPORT_REFS       = True
INCLUDE_EMB_RGCN  = True   # False halves the GPU cost (20 runs instead of 40)
PROBE_ENCODER     = True
PROBE_EMB_LEARNS  = True   # assert the embedding table actually receives gradient
STOP_AFTER_N_RUNS = None

print('split  :', SPLIT)
print('seeds  : %d -> %s' % (len(SEEDS), SEEDS))
print('results:', ABL_ROOT.relative_to(REPO_ROOT))

---
## 3 - The encoders

`MLPLayer` is copied verbatim from `ablation_encoder_swap.ipynb` so the `emb_mlp` arm differs from the
published `mlp` arm in **exactly one** respect: where node features come from.

`EmbeddingMixin` installs the learnable table. It does not reimplement `HeteroRGCN.forward` - it binds
the live `nn.Parameter` onto `G[ntype].inp` at the top of every forward and then delegates to the
library. That matters for correctness: `TxGNN.pretrain` passes `self.G.to(self.device)` (a *moved copy*
of the graph) into `forward_minibatch`, and `finetune` reassigns `self.G`, so any binding done once at
construction time would be silently replaced by a detached tensor and the table would never train.
Binding per forward is immune to both. `PROBE_EMB_LEARNS` verifies it empirically rather than trusting
the argument.

In [ ]:
class MLPLayer(nn.Module):
    '''No message passing: h[ntype] = W x[ntype]. Verbatim from ablation_encoder_swap.ipynb.'''

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = nn.Linear(in_size, out_size)
        self.in_size, self.out_size = in_size, out_size

    def forward(self, G, feat_dict):
        return {nt: self.weight(feat_dict[nt]) for nt in G.node_types}


_TXGNN_MODULE    = sys.modules['txgnn.TxGNN']
_ORIG_HETERORGCN = HeteroRGCN
LAYER_CLASSES    = {'rgcn': HeteroRGCNLayer, 'mlp': MLPLayer}


def make_encoder_class(layer_kind, learnable_emb, kind):
    layer_cls = LAYER_CLASSES[layer_kind]

    class SwappedHeteroRGCN(_ORIG_HETERORGCN):
        '''HeteroRGCN with an optional layer swap and an optional learnable node-embedding table.'''

        def __init__(self, G, **kw):
            super().__init__(G, **kw)
            if layer_kind != 'rgcn':
                self.layer1 = layer_cls(kw['in_size'], kw['hidden_size'])
                self.layer2 = layer_cls(kw['hidden_size'], kw['out_size'])
            self.encoder_kind  = kind
            self.learnable_emb = learnable_emb
            self.node_emb, self._emb_key = None, {}
            if learnable_emb:
                self.node_emb = nn.ParameterDict()
                for nt in G.node_types:
                    k = nt.replace('/', '__')          # ParameterDict keys cannot contain '.'
                    self._emb_key[nt] = k
                    p = nn.Parameter(torch.empty(G[nt].num_nodes, kw['in_size']))
                    nn.init.xavier_uniform_(p)         # same init as initialize_node_embedding
                    self.node_emb[k] = p

        def _bind_embeddings(self, G):
            '''Point G[ntype].inp at the live Parameters. Must run inside every forward - see cell 5.'''
            if not self.learnable_emb:
                return
            for nt in G.node_types:
                G[nt].inp = self.node_emb[self._emb_key[nt]]

        def forward(self, G, *a, **kw):
            self._bind_embeddings(G)
            return super().forward(G, *a, **kw)

        def forward_minibatch(self, pos_G, neg_G, G, *a, **kw):
            self._bind_embeddings(G)
            return super().forward_minibatch(pos_G, neg_G, G, *a, **kw)

    SwappedHeteroRGCN.__name__ = 'HeteroRGCN_' + kind
    return SwappedHeteroRGCN


ENCODER_SPECS = {
    'rgcn':     dict(layer='rgcn', emb=False),
    'mlp':      dict(layer='mlp',  emb=False),
    'emb_mlp':  dict(layer='mlp',  emb=True),
    'emb_rgcn': dict(layer='rgcn', emb=True),
}


def install_encoder(kind):
    spec = ENCODER_SPECS[kind]
    cls = (_ORIG_HETERORGCN if kind == 'rgcn'
           else make_encoder_class(spec['layer'], spec['emb'], kind))
    _TXGNN_MODULE.HeteroRGCN = cls
    return cls


install_encoder('rgcn')
print('encoders:', sorted(ENCODER_SPECS))
print('txgnn.TxGNN.HeteroRGCN ->', _TXGNN_MODULE.HeteroRGCN.__name__)

---
## 4 - The arms

In [ ]:
ARMS = [
    {'arm': 'rgcn',     'encoder': 'rgcn',    'note': 'reference: published model (imported)'},
    {'arm': 'mlp',      'encoder': 'mlp',     'note': 'reference: frozen-feature MLP (imported)'},
    {'arm': 'emb_mlp',  'encoder': 'emb_mlp', 'note': 'THE CONTROL: learnable table, no message passing'},
]
if INCLUDE_EMB_RGCN:
    ARMS.append({'arm': 'emb_rgcn', 'encoder': 'emb_rgcn',
                 'note': 'does the published model also suffer from frozen features?'})

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
for _r in (REF_ARM, REF_ARM_2):
    assert _r in ARMS_BY_NAME, 'reference arm %r missing' % _r

_new = [a for a in ARMS if a['arm'] not in (REF_ARM, REF_ARM_2)]
print('%d arms x %d seeds | %d need GPU time (2 reference arms import free)\n'
      % (len(ARMS), len(SEEDS), len(_new) * len(SEEDS)))
for a in ARMS:
    sp = ENCODER_SPECS[a['encoder']]
    print('  %-10s layer=%-5s message passing=%-4s learnable features=%-5s'
          % (a['arm'], sp['layer'], sp['layer'] != 'mlp', sp['emb']))
    print('  %-10s %s' % ('', a['note']))

---
## 5 - Mechanics

`encoder_reads_graph` is the runtime proof that the `mlp`-based arms pass no messages: it re-runs the
encoder with every `edge_index` emptied and reports the largest change in node embeddings. It must be
**0.0** for `emb_mlp` and non-zero for `emb_rgcn`.

`embedding_learns` is the proof that the table is actually trained: it compares the embedding tensor
before and after the first optimiser step of fine-tuning. A table that never moved would make the arm
silently identical to a random-feature MLP, which is the exact failure this notebook exists to avoid.

In [ ]:
class RunData:
    '''Minimal TxData stand-in. Graph rebuilt per run; nothing filtered in this notebook.'''

    def __init__(self, base, data_folder, split):
        self.df       = base['df']
        self.df_train = base['df_train']
        self.df_valid = base['df_valid']
        self.df_test  = base['df_test']
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


_SEED_CACHE = {}


def get_seed_data(seed):
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def node_emb_params(model):
    ne = getattr(model, 'node_emb', None)
    return 0 if ne is None else sum(p.numel() for p in ne.parameters())


def encoder_reads_graph(model, G):
    '''Max |delta| in node embeddings when every edge_index is emptied. 0.0 => no message passing.'''
    model.eval()
    saved = {et: G[et].edge_index for et in G.edge_types}
    empty = torch.zeros((2, 0), dtype=torch.long, device=saved[next(iter(saved))].device)
    try:
        with torch.no_grad():
            h_full = model(G, G, return_h=True)
            for et in G.edge_types:
                G[et].edge_index = empty
            h_none = model(G, G, return_h=True)
        return float(max((h_full[nt] - h_none[nt]).abs().max().item() for nt in h_full))
    finally:
        for et, ei in saved.items():
            G[et].edge_index = ei


def embedding_learns(m):
    '''Run ONE finetune epoch and report the largest change in the embedding table.

    Called before the real training run, on a throwaway copy of the optimiser state, so it costs one
    epoch out of 1000. Returns None when the arm has no table.
    '''
    model = m.model
    if getattr(model, 'node_emb', None) is None:
        return None
    before = {k: p.detach().clone() for k, p in model.node_emb.items()}
    m.finetune(n_epoch=1, learning_rate=FINETUNE_CFG['learning_rate'],
               train_print_per_n=10 ** 9, valid_per_n=10 ** 9)
    return float(max((model.node_emb[k].detach() - v).abs().max().item()
                     for k, v in before.items()))

In [ ]:
# --- import the two reference arms ----------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_ref(arm_name, seed):
    if not IMPORT_REFS:
        return False, 'disabled'
    return import_matching_run(
        SWAP_ROOT / ('%s_seed%d' % (arm_name, seed)), run_dir(arm_name, seed),
        expect={'split': SPLIT, 'seed': seed, 'dropped_edges_total': 0, 'proto': True,
                'encoder': arm_name,
                'signature_disease_etypes': SIG_ETYPES_DEFAULT,
                'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
                'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG},
        extra={'experiment': 'learnable_embeddings', 'learnable_emb': False,
               'node_emb_params': 0})


if IMPORT_REFS:
    for _a in (REF_ARM, REF_ARM_2):
        n_ok = 0
        for s in SEEDS:
            if is_done(_a, s):
                n_ok += 1
                continue
            ok, why = try_import_ref(_a, s)
            n_ok += bool(ok)
            if not ok:
                print('[import] %s_seed%d: will train (%s)' % (_a, s, why))
        print('[import] %-9s %d/%d available' % (_a, n_ok, len(SEEDS)))

# --- plan -----------------------------------------------------------------------------------------
_times = []
for _p in sorted(SWAP_ROOT.glob('rgcn_seed*/run_meta.json')):
    _r = json.loads(_p.read_text())
    if _r.get('pretrain_seconds') and _r.get('finetune_seconds'):
        _times.append(_r['pretrain_seconds'] + _r['finetune_seconds'])
EST_MIN = (float(np.mean(_times)) / 60) if _times else 24.6

rows = []
for a in ARMS:
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    todo = len(SEEDS) - len(done)
    est = 0.0 if a['arm'] in (REF_ARM, REF_ARM_2) else todo * EST_MIN / 60
    rows.append({'arm': a['arm'], 'learnable features': ENCODER_SPECS[a['encoder']]['emb'],
                 'done': len(done), 'todo': todo, 'est GPU-h': round(est, 2)})
display(pd.DataFrame(rows).style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]
pending = [k for k in RUN_KEYS if not is_done(*k)]
print('\n%d run_keys, %d done, %d pending | ~%.1f GPU-hours at %.1f min/run'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending),
         len(pending) * EST_MIN / 60, EST_MIN))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))

---
## 6 - The grid

In [ ]:
def run_one(arm_name, seed, verbose=True):
    arm  = ARMS_BY_NAME[arm_name]
    kind = arm['encoder']
    spec = ENCODER_SPECS[kind]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name in (REF_ARM, REF_ARM_2):
        ok, why = try_import_ref(arm_name, seed)
        if ok:
            print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
            return pd.read_csv(el_csv)

    print('\n' + '#' * 78)
    print('# %s   seed %d   layer=%s   message passing=%s   learnable features=%s'
          % (arm_name, seed, spec['layer'], spec['layer'] != 'mlp', spec['emb']))
    print('#' * 78)

    base = get_seed_data(seed)
    set_all_seeds(seed)
    d = RunData(base, DATA_FOLDER, SPLIT)
    rel_names = {et[1] for et in d.G.edge_types}
    assert all(r in rel_names for r in SIG_ETYPES_DEFAULT), 'KG is not intact'
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    print('[graph] train edges %d | edge types %d (KG intact in every arm)'
          % (len(d.df_train), len(etypes_after)))

    installed = install_encoder(kind)

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**MODEL_CFG)

    assert type(m.model) is installed, 'encoder swap did not take: %s' % type(m.model).__name__
    layer_cls = type(m.model.layer1).__name__
    assert layer_cls == LAYER_CLASSES[spec['layer']].__name__, \
        'layer1 is %s, expected %s' % (layer_cls, LAYER_CLASSES[spec['layer']].__name__)

    enc_p, emb_p, tot_p = encoder_params(m.model), node_emb_params(m.model), get_n_params(m.model)
    assert (emb_p > 0) == spec['emb'], \
        'learnable_emb=%s but node_emb_params=%d' % (spec['emb'], emb_p)
    n_nodes = sum(d.G[nt].num_nodes for nt in d.G.node_types)
    if spec['emb']:
        assert emb_p == n_nodes * MODEL_CFG['n_inp'], \
            'embedding table is %d, expected %d x %d' % (emb_p, n_nodes, MODEL_CFG['n_inp'])
    print('[model] %s | layer=%s | encoder %s | node table %s | total %s'
          % (type(m.model).__name__, layer_cls, format(enc_p, ','), format(emb_p, ','),
             format(tot_p, ',')))

    # ---- the table must actually train; costs 1 of 1000 finetune epochs ----
    emb_delta = None
    if PROBE_EMB_LEARNS and spec['emb']:
        set_all_seeds(seed)
        emb_delta = embedding_learns(m)
        print('[probe] embedding table moved by %.3g after one epoch' % emb_delta)
        assert emb_delta > 0, 'the embedding table received no gradient - it is not being trained'
        # rebuild from scratch so the probe cannot contaminate the measured run
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        set_all_seeds(seed)
        m = TxGNN(data=d, weight_bias_track=False,
                  exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
        m.model_initialize(**MODEL_CFG)

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    graph_sensitivity = None
    if PROBE_ENCODER:
        graph_sensitivity = encoder_reads_graph(m.best_model, m.G)
        print('[probe] max |delta h| with all edges removed: %.6g' % graph_sensitivity)
        if spec['layer'] == 'mlp':
            assert graph_sensitivity < 1e-6, 'mlp arm still reads the graph (%g)' % graph_sensitivity
        else:
            assert graph_sensitivity > 1e-6, 'rgcn arm does not read the graph (%g)' % graph_sensitivity

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'learnable_embeddings',
        'note': arm['note'],
        'encoder': kind, 'encoder_class': type(m.model).__name__, 'layer_class': layer_cls,
        'message_passing': spec['layer'] != 'mlp',
        'learnable_emb': spec['emb'],
        'graph_sensitivity': graph_sensitivity,
        'emb_delta_after_1_epoch': emb_delta,
        'encoder_params': enc_p, 'node_emb_params': emb_p, 'n_params': tot_p,
        'n_nodes_total': n_nodes,
        'proto': MODEL_CFG['proto'],
        'signature_disease_etypes': SIG_ETYPES_DEFAULT,
        'dropped_edges_total': 0,
        'train_edges': len(d.df_train),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_learnable_embeddings' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication %.4f | contraindication %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    del m, d, auroc_rel, auprc_rel
    install_encoder('rgcn')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]
    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 7 - Aggregate

Two reference arms, so every arm gets two paired deltas: against `rgcn` (the published model) and
against `mlp` (the frozen-feature MLP).

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'learnable_emb': meta.get('learnable_emb'),
                         'message_passing': meta.get('message_passing'),
                         'encoder_params': meta.get('encoder_params'),
                         'node_emb_params': meta.get('node_emb_params'),
                         'n_params': meta.get('n_params'),
                         'graph_sensitivity': meta.get('graph_sensitivity'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}
KNOWN_P = {'rgcn': 1010000, 'mlp': 20200, 'emb_mlp': 20200, 'emb_rgcn': 1010000}

long = _load_all(ABL_ROOT)
summary = wide = ref = ref2 = None

if long.empty:
    print('No runs on disk yet - run section 6 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))
    _first = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
    info = long.groupby('arm').agg(encoder_params=('encoder_params', _first),
                                   node_emb_params=('node_emb_params', _first),
                                   n_params=('n_params', _first),
                                   learnable_emb=('learnable_emb', 'first'),
                                   message_passing=('message_passing', 'first'))
    present = set(wide.index.get_level_values('arm'))
    ref  = wide.xs(REF_ARM,   level='arm') if REF_ARM   in present else None
    ref2 = wide.xs(REF_ARM_2, level='arm') if REF_ARM_2 in present else None

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        ep = info.loc[arm, 'encoder_params']
        rec = {'arm': arm, 'n_seeds': len(g),
               'learnable feat.': info.loc[arm, 'learnable_emb'],
               'msg passing': info.loc[arm, 'message_passing'],
               'encoder_params': KNOWN_P.get(arm, np.nan) if pd.isna(ep) else ep,
               'node_emb_params': info.loc[arm, 'node_emb_params']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            for tag, r in (('', ref), ('_vs_mlp', ref2)):
                if r is None:
                    continue
                paired = (g[key] - r[key]).dropna()
                rec['%s_delta%s' % (key, tag)] = paired.mean() if len(paired) else np.nan
                rec['%s_delta%s_CI' % (key, tag)] = _ci95(paired)
                rec['%s_better%s' % (key, tag)] = int((paired > 0).sum())
                rec['%s_n_paired%s' % (key, tag)] = len(paired)
        out.append(rec)

    order = [a['arm'] for a in ARMS]
    summary = (pd.DataFrame(out)
                 .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(order)}))
                 .sort_values('_o').drop(columns='_o').reset_index(drop=True))
    summary.to_csv(ABL_ROOT / 'learnable_embeddings_summary.csv', index=False)

    cols = [c for c in ['arm', 'n_seeds', 'learnable feat.', 'msg passing', 'encoder_params',
                        'node_emb_params', 'indication_AUPRC', 'indication_CI',
                        'indication_delta', 'indication_delta_vs_mlp',
                        'contraindication_AUPRC', 'contraindication_delta',
                        'contraindication_delta_vs_mlp', 'micro_AUPRC', 'micro_delta']
            if c in summary.columns]
    print('%d arms, %d runs -> %s\n'
          % (len(summary), len(wide),
             (ABL_ROOT / 'learnable_embeddings_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style
            .format('{:.4f}', subset=[c for c in cols if c.endswith(('AUPRC', 'delta', 'CI', '_mlp'))],
                    na_rep='-')
            .format({'encoder_params': '{:,.0f}', 'node_emb_params': '{:,.0f}'}, na_rep='-')
            .hide(axis='index'))

### 7.1 - The verdict

The chance floor is **0.500** (one sampled negative per positive). `mlp` sits at 0.5285 indication and
`rgcn` at 0.8318, so the gap `emb_mlp` has to close is 0.303.

In [ ]:
FLOOR = 0.500
if summary is None or ref is None or ref2 is None or 'emb_mlp' not in set(summary.arm):
    print('Need finished rgcn, mlp and emb_mlp runs.')
else:
    s = summary.set_index('arm')
    print('=' * 78)
    print('%-18s %-10s %-10s %-10s %s' % ('', 'indication', 'contraind.', 'micro', 'n'))
    for arm in [a['arm'] for a in ARMS if a['arm'] in s.index]:
        print('%-18s %-10.4f %-10.4f %-10.4f %d'
              % (arm, s.loc[arm, 'indication_AUPRC'], s.loc[arm, 'contraindication_AUPRC'],
                 s.loc[arm, 'micro_AUPRC'], s.loc[arm, 'n_seeds']))
    print('=' * 78)

    gap = s.loc[REF_ARM, 'indication_AUPRC'] - s.loc[REF_ARM_2, 'indication_AUPRC']
    rec = s.loc['emb_mlp', 'indication_AUPRC'] - s.loc[REF_ARM_2, 'indication_AUPRC']
    print('\nindication: rgcn %.4f, mlp %.4f, gap %.4f' % (
        s.loc[REF_ARM, 'indication_AUPRC'], s.loc[REF_ARM_2, 'indication_AUPRC'], gap))
    print('            emb_mlp %.4f -> recovers %+.4f = %.0f%% of the gap'
          % (s.loc['emb_mlp', 'indication_AUPRC'], rec, 100 * rec / gap if gap else float('nan')))
    hr = lambda v: 100 * (v - FLOOR) / (s.loc[REF_ARM, 'indication_AUPRC'] - FLOOR)
    print('            above-chance margin retained: mlp %.1f%%, emb_mlp %.1f%%'
          % (hr(s.loc[REF_ARM_2, 'indication_AUPRC']), hr(s.loc['emb_mlp', 'indication_AUPRC'])))

    frac = rec / gap if gap else float('nan')
    print('\nverdict:')
    if frac < 0.25:
        print('  NODE IDENTITY IS NOT THE ISSUE. Even with %s free parameters and no frozen features,'
              % format(int(s.loc['emb_mlp', 'node_emb_params']), ','))
        print('  the MLP stays near the floor. Message passing is genuinely load-bearing for')
        print('  indication, and the encoder-swap conclusion stands as published.')
    elif frac > 0.75:
        print('  THE COLLAPSE WAS THE FROZEN FEATURES. A plain embedding table recovers most of the')
        print('  gap with no message passing at all, so the R-GCN is largely serving as a way to give')
        print('  nodes identities. Note the comparison is NOT at equal capacity - say so explicitly.')
    else:
        print('  PARTIAL (%.0f%%). Both effects are real: frozen features cost the mlp arm something,'
              % (100 * frac))
        print('  and message passing still buys the rest. Report both numbers, neither alone.')

    if 'emb_rgcn' in s.index:
        d = s.loc['emb_rgcn', 'indication_delta']
        print('\nemb_rgcn vs rgcn: %+.4f indication -> the published model %s limited by its frozen'
              ' features.' % (d, 'IS also' if d > 0.01 else 'is NOT materially'))

In [ ]:
# Per-seed paired deltas against both references.
if summary is not None:
    for label, r in ((REF_ARM, ref), (REF_ARM_2, ref2)):
        if r is None:
            continue
        for key in ['indication', 'contraindication']:
            arms = [a for a in summary.arm
                    if a in wide.index.get_level_values('arm') and a != label]
            if not arms:
                continue
            tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - r[key]) for a in arms}).T
            seeds_present = list(tbl.columns)
            tbl.columns = ['seed %d' % x for x in seeds_present]
            tbl['mean'] = tbl.mean(axis=1)
            tbl['better'] = (tbl[['seed %d' % x for x in seeds_present]] > 0).sum(axis=1)
            print('\npaired delta AUPRC vs %s - %s  (positive = the arm is better)' % (label, key))
            display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'better'],
                                     na_rep='-'))

---
## 8 - What this changes in the write-up

Whatever the outcome, this notebook **removes the main caveat** in section 11 of
`docs/ABLATION_FINDINGS.md` and section 1.7 of the comment-paper draft. Replace the sentence

> Until it exists, "the GNN is unnecessary" is not testable from this data.

with the measured result from section 7.1.

### Still open after this notebook

1. **Capacity is not matched and cannot be.** 12.9M embedding parameters vs the R-GCN's 1.01M. A
   capacity-matched variant would need a low-dimensional table (e.g. 8 dims x 129k nodes ~ 1.03M),
   which is a clean follow-up if `emb_mlp` recovers the gap: it would separate "nodes need identities"
   from "nodes need 100-dimensional identities".
2. **One split protocol and the lenient metric** - see `ablation_full_ranking.ipynb`.
3. **Pretraining is held at one epoch.** A learnable table may be much more pretraining-hungry than
   frozen features, so a `pretrain n_epoch=0` version of `emb_mlp` would bound how much of any recovery
   comes from the self-supervised pass. See `ablation_pretrain_and_gate.ipynb`.
4. **Transductive by construction.** An embedding table cannot generalise to a node unseen at training
   time. That is fine here - the split is zero-shot over *drug-disease edges*, not over nodes - but it
   means this arm is not a deployable architecture, only a diagnostic.